# Download Data

This example notebook shows you how to download water data time-series from Google Earthengine for 
1. Annual JRC surface water data
2. Dynamic World - monthly aggregated data

### Imports

In [ ]:
from water_timeseries.downloader import EarthEngineDownloader
import geopandas as gpd

#### Loading vector dataset
* we are loading the provided lake polygon dataset
* the specific unique lake id is "id_geohash"
* explore the dataset

In [ ]:
vector_path = "lake_polygons.parquet"
gdf = gpd.read_parquet(vector_path)

In [ ]:
gdf.head()

In [ ]:
gdf.explore(width=1200, height=600)

#### Setup GEE account
* set project name to EE_PROJECT environment variable (preferred)

In [ ]:
import os
ee_project = "YOUR_GEE_PROJECT"
os.environ["EE_PROJECT"] = ee_project

#### Setup logger (optional)
* here I am using loguru, you can also choose others

In [ ]:
import loguru

logger = loguru.logger
logger.add("logfile_dl.log", level=0)

#### Download JRC dataset
* requires polygon vector dataset + GEE account
* returns xarray dataset

#### Simple case JRC
* Download JRC annual timeseries for
  * default years (2000-2021)
  * all polygons
  * serially (if too large, polygon dataset will be chunked to avoid GEE Errors)
  * takes ~ 20 seconds
* This returns a xr.Dataset with
  * 2 coordinates: date, id_geohash (or your specific selection)
  * 4 variables: area_land, area_nodata, area_water_permanent, area_water_seasonal      

In [ ]:
dl = EarthEngineDownloader(ee_project=ee_project, ee_auth=True, logger=logger)
ds_jrc = dl.download_jrc_annual(
    vector_dataset=vector_path,
    name_attribute="id_geohash",
    n_parallel=1,
)

In [ ]:
ds_jrc

#### Save to netCDF file

In [ ]:
ds_jrc.to_netcdf("jrc_annual.nc")

#### Simple case Dynamic World
* Download Dynamic World monthly timeseries for
  * default years (2016-2025)
  * all polygons
  * serially (if too large, polygon dataset will be chunked to avoid GEE Errors)
  * takes ~ 20 seconds
* This returns a xr.Dataset with
  * 2 coordinates: date, id_geohash (or your specific selection)
  * 4 variables: bare, built, crops, flooded_vegetation, grass, shrub_and_scrub, snow_and_ice, trees, water

In [ ]:
ds_dw = dl.download_dw_monthly(
    vector_dataset=vector_path,
    name_attribute="id_geohash",
    years=range(2017, 2027),
    months=[6, 7, 8, 9],
    n_parallel=2,
)

#### Save to netCDF file

In [ ]:
ds_dw.to_netcdf("dw_monthly.nc")